In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
import joblib
from sklearn.preprocessing import StandardScaler
from io import StringIO
from tqdm import tqdm

## Helper Classes

In [ ]:
class AnomalyPreprocessor:
    def __init__(self, str_bucket, str_dirname_output):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.s3_client = boto3.client('s3')
        self.df_data = None
        self.df_train = None
        self.df_valid = None
        self.df_test = None
        self.df_train_processed = None
        self.df_valid_processed = None
        self.df_test_processed = None
        self.scaler = None
        self.list_feature_cols = None

    def import_data(self):
        """Load cleaned data from S3."""
        print(f"Loading data from S3: s3://{self.str_bucket}/00_data_collection/paysim_clean.csv")
        str_uri = f's3://{self.str_bucket}/00_data_collection/paysim_clean.csv'
        self.df_data = pd.read_csv(str_uri)
        print(f"Data loaded: {self.df_data.shape}")
        return self.df_data

    def feature_engineering(self):
        """Create domain-relevant features for anomaly detection."""
        if self.df_data is None:
            raise ValueError("Data not loaded. Call import_data() first.")
        
        print("\nPerforming feature engineering...")
        
        # Balance changes
        self.df_data['balance_change_orig'] = self.df_data['newbalanceOrig'] - self.df_data['oldbalanceOrg']
        self.df_data['balance_change_dest'] = self.df_data['newbalanceDest'] - self.df_data['oldbalanceDest']
        
        # Amount to balance ratio (risk indicator)
        self.df_data['amount_to_balance_ratio'] = self.df_data['amount'] / (self.df_data['oldbalanceOrg'].abs() + 1)
        
        # Balance zeroing (strong fraud signal)
        self.df_data['is_balance_zeroed'] = (self.df_data['newbalanceOrig'] == 0).astype(int)
        
        # Hour of day (temporal feature)
        self.df_data['hour_of_day'] = self.df_data['step'] % 24
        
        # One-hot encode transaction type
        df_type_encoded = pd.get_dummies(self.df_data['type'], prefix='type')
        self.df_data = pd.concat([self.df_data, df_type_encoded], axis=1)
        
        print(f"New features created. Shape now: {self.df_data.shape}")
        print(f"New columns: balance_change_orig, balance_change_dest, amount_to_balance_ratio, is_balance_zeroed, hour_of_day, {list(df_type_encoded.columns)}")
        
        return self.df_data

    def split_data(self):
        """Perform temporal train/valid/test split: 60% train (non-fraud only), 20% valid (all), 20% test (all)."""
        if self.df_data is None:
            raise ValueError("Data not loaded.")
        
        print("\nPerforming temporal train/valid/test split...")
        
        int_max_step = self.df_data['step'].max()
        int_train_threshold = int(int_max_step * 0.6)
        int_valid_threshold = int(int_max_step * 0.8)
        print(f"Train split point (step): {int_train_threshold}")
        print(f"Valid split point (step): {int_valid_threshold}")
        
        # Training: first 60% of steps, non-fraud only
        df_train_period = self.df_data[self.df_data['step'] <= int_train_threshold]
        self.df_train = df_train_period[df_train_period['isFraud'] == 0].copy()
        
        # Validation: next 20% of steps, all data
        self.df_valid = self.df_data[
            (self.df_data['step'] > int_train_threshold) & (self.df_data['step'] <= int_valid_threshold)
        ].copy()
        
        # Testing: last 20% of steps, all data
        self.df_test = self.df_data[self.df_data['step'] > int_valid_threshold].copy()
        
        print(f"Training set: {self.df_train.shape} (non-fraud only from first 60% of steps)")
        print(f"Validation set: {self.df_valid.shape} (all transactions from next 20% of steps)")
        print(f"  - Non-fraud in valid: {(self.df_valid['isFraud'] == 0).sum():,}")
        print(f"  - Fraud in valid: {(self.df_valid['isFraud'] == 1).sum():,} ({self.df_valid['isFraud'].mean()*100:.4f}%)")
        print(f"Testing set: {self.df_test.shape} (all transactions from last 20% of steps)")
        print(f"  - Non-fraud in test: {(self.df_test['isFraud'] == 0).sum():,}")
        print(f"  - Fraud in test: {(self.df_test['isFraud'] == 1).sum():,} ({self.df_test['isFraud'].mean()*100:.4f}%)")
        
        return self.df_train, self.df_valid, self.df_test

    def scale_features(self):
        """Standardize numeric features using training data statistics."""
        if self.df_train is None or self.df_valid is None or self.df_test is None:
            raise ValueError("Data not split. Call split_data() first.")
        
        print("\nScaling numeric features...")
        
        # Define features to scale (exclude identifiers and target)
        list_exclude = ['step', 'type', 'nameOrig', 'nameDest', 'isFraud', 'isFlaggedFraud']
        self.list_feature_cols = [col for col in self.df_train.columns if col not in list_exclude and self.df_train[col].dtype in [np.float64, np.int64]]
        
        print(f"Features to scale ({len(self.list_feature_cols)}): {self.list_feature_cols}")
        
        # Fit scaler on training data
        self.scaler = StandardScaler()
        self.scaler.fit(self.df_train[self.list_feature_cols])
        
        # Transform train, valid, and test
        arr_train_scaled = self.scaler.transform(self.df_train[self.list_feature_cols])
        arr_valid_scaled = self.scaler.transform(self.df_valid[self.list_feature_cols])
        arr_test_scaled = self.scaler.transform(self.df_test[self.list_feature_cols])
        
        # Create processed dataframes
        self.df_train_processed = pd.DataFrame(arr_train_scaled, columns=self.list_feature_cols, index=self.df_train.index)
        self.df_valid_processed = pd.DataFrame(arr_valid_scaled, columns=self.list_feature_cols, index=self.df_valid.index)
        self.df_test_processed = pd.DataFrame(arr_test_scaled, columns=self.list_feature_cols, index=self.df_test.index)
        
        # Keep target and identifiers
        self.df_train_processed['isFraud'] = self.df_train['isFraud'].values
        self.df_valid_processed['isFraud'] = self.df_valid['isFraud'].values
        self.df_test_processed['isFraud'] = self.df_test['isFraud'].values
        
        print(f"Scaled training set: {self.df_train_processed.shape}")
        print(f"Scaled validation set: {self.df_valid_processed.shape}")
        print(f"Scaled test set: {self.df_test_processed.shape}")
        
        return self.df_train_processed, self.df_valid_processed, self.df_test_processed

    def save_splits(self):
        """Save preprocessed splits and scaler to S3."""
        if self.df_train_processed is None or self.df_valid_processed is None or self.df_test_processed is None:
            raise ValueError("Data not processed. Call scale_features() first.")
        
        print("\nSaving splits to S3...")
        
        # Save locally and to S3
        str_train_path = os.path.join(self.str_dirname_output, 'train_processed.csv')
        str_valid_path = os.path.join(self.str_dirname_output, 'valid_processed.csv')
        str_test_path = os.path.join(self.str_dirname_output, 'test_processed.csv')
        str_scaler_path = os.path.join(self.str_dirname_output, 'scaler.pkl')
        str_feature_cols_path = os.path.join(self.str_dirname_output, 'feature_cols.pkl')
        
        self.df_train_processed.to_csv(str_train_path, index=False)
        self.df_valid_processed.to_csv(str_valid_path, index=False)
        self.df_test_processed.to_csv(str_test_path, index=False)
        joblib.dump(self.scaler, str_scaler_path)
        joblib.dump(self.list_feature_cols, str_feature_cols_path)
        
        try:
            self.s3_client.upload_file(str_train_path, self.str_bucket, '02_preprocessing/train_processed.csv')
            self.s3_client.upload_file(str_valid_path, self.str_bucket, '02_preprocessing/valid_processed.csv')
            self.s3_client.upload_file(str_test_path, self.str_bucket, '02_preprocessing/test_processed.csv')
            self.s3_client.upload_file(str_scaler_path, self.str_bucket, '02_preprocessing/scaler.pkl')
            self.s3_client.upload_file(str_feature_cols_path, self.str_bucket, '02_preprocessing/feature_cols.pkl')
            print("Uploaded to S3 successfully")
        except Exception as e:
            print(f"Error uploading to S3: {e}")
            print(f"Files saved locally at: {self.str_dirname_output}")

## Constants

In [ ]:
str_bucket = 'anomaly-detection-demo-repo'
str_task = 'preprocessing'
str_dirname_output = './output'

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass
print(f"Output directory ready: {str_dirname_output}")

## Load Data

In [ ]:
preprocessor = AnomalyPreprocessor(str_bucket, str_dirname_output)
df_data = preprocessor.import_data()

## Feature Engineering

In [ ]:
df_engineered = preprocessor.feature_engineering()

In [ ]:
print("\nFeature engineering results:")
print(df_engineered[['balance_change_orig', 'balance_change_dest', 'amount_to_balance_ratio', 'is_balance_zeroed', 'hour_of_day']].describe())

## Train/Valid/Test Split

In [ ]:
df_train, df_valid, df_test = preprocessor.split_data()

## Feature Scaling

In [ ]:
df_train_processed, df_valid_processed, df_test_processed = preprocessor.scale_features()

In [ ]:
print("\nScaled training set (first 5 rows):")
print(df_train_processed.head())
print(f"\nScaled training set statistics:")
print(df_train_processed.describe())

## Save Preprocessed Data

In [ ]:
preprocessor.save_splits()

## Completion

In [ ]:
print("\n=== PREPROCESSING COMPLETE ===")
print(f"Training set: {df_train_processed.shape}")
print(f"Validation set: {df_valid_processed.shape}")
print(f"Test set: {df_test_processed.shape}")
print(f"Features: {preprocessor.list_feature_cols}")